# 03 — Training and Inference Pipeline

Run a minimal end-to-end training and inference loop for ContextQuantFusionNet.

In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, random_split

torch.manual_seed(101)

In [ ]:
class ContextQuantFusionNet(nn.Module):
    def __init__(self, price_features=2, text_features=2, hidden=32, n_classes=5):
        super().__init__()
        self.temporal = nn.LSTM(price_features, hidden, batch_first=True)
        self.text_branch = nn.Sequential(nn.Linear(text_features, hidden), nn.ReLU())
        self.fusion = nn.Sequential(nn.Linear(hidden * 2, hidden), nn.ReLU(), nn.Linear(hidden, n_classes))

    def forward(self, price_seq, text_vec):
        t, _ = self.temporal(price_seq)
        t_last = t[:, -1, :]
        txt = self.text_branch(text_vec)
        return self.fusion(torch.cat([t_last, txt], dim=1))

In [ ]:
n = 1200
X_price = torch.randn(n, 20, 2)
X_text = torch.randn(n, 2)
raw = X_price[:, :, 0].mean(dim=1) + 0.6 * X_text[:, 0]
y = torch.bucketize(raw, boundaries=torch.tensor([-0.8, -0.2, 0.2, 0.8]))  # 0..4

dataset = TensorDataset(X_price, X_text, y)
train_ds, val_ds = random_split(dataset, [1000, 200])
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128)

In [ ]:
model = ContextQuantFusionNet()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(1, 16):
    model.train()
    for p, t, target in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(p, t), target)
        loss.backward()
        optimizer.step()

    if epoch % 5 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            vp, vt, vy = next(iter(val_loader))
            pred = model(vp, vt).argmax(dim=1)
            acc = (pred == vy).float().mean().item()
        print(f'Epoch {epoch:>2} | val_batch_acc={acc:.3f}')

In [ ]:
model.eval()
with torch.no_grad():
    sample_price = torch.randn(1, 20, 2)
    sample_text = torch.tensor([[0.7, 0.9]], dtype=torch.float32)  # positive/high-confidence sentiment
    probs = torch.softmax(model(sample_price, sample_text), dim=1).squeeze()

labels = ['Strong Sell', 'Sell', 'Hold', 'Buy', 'Strong Buy']
for l, p in zip(labels, probs.tolist()):
    print(f'{l:>11}: {p:.3f}')
print('Predicted class:', labels[int(probs.argmax())])

## Exercises
1. Replace synthetic data with your real aligned dataset.
2. Add checkpoint save/load with `torch.save` and `torch.load`.
3. Compute full validation accuracy over all batches.